# SAEB: da fonte oficial ao Power BI

Demonstracao da serie 2007-2023 de proficiencias oficiais agregadas por UF. O fluxo distingue a ingestao historica e a verificacao especifica dos resultados agregados de 2023. A saida e `data/gold/fatos/fato_saeb.parquet`.

## Ciclo

Resultados oficiais e materiais de auditoria na Raw -> auditoria de estruturas e categorias -> duas rotas Bronze para 2023 -> Silver harmonizada -> fato Gold -> Power BI. A Gold preserva os valores oficiais no grao `ANO + UF + ETAPA + REDE + DISCIPLINA`.

## Criterios de tratamento e linhagem

O SAEB exige regras por edicao porque nomes, cabecalhos e codigos mudam. Em 2007 e 2009, a fonte por UF oferece `Total - Estadual e Municipal`; essa limitacao historica e preservada em `REDE_ORIGEM`. Em 2011, o agregado publico e identificado pelo codigo oficial `ID_TIPO_REDE = 5`, com localizacao e capital iguais a zero. De 2013 a 2023, utiliza-se `Total - Federal, Estadual e Municipal`, com localizacao e capital `Total`. Nenhum total que inclua rede privada e usado.

As proficiencias de Lingua Portuguesa (`LP`) e Matematica (`MT`) do 5o e 9o anos sao harmonizadas como `ANOS_INICIAIS` e `ANOS_FINAIS`. Em 2015, zero significa media nao calculada naquele estrato e e convertido em ausencia somente nessa edicao.

A Silver preserva rede, localizacao, capital, arquivo, aba, linha, coluna e granularidade de origem. A Gold entrega somente o grao analitico e nao recompõe medias a partir de escolas. O total esperado e `9 edicoes x 27 UFs x 2 etapas x 2 disciplinas = 972`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
DIRETORIO_ATUAL = Path.cwd().resolve()
ROOT = next((p for p in (DIRETORIO_ATUAL, *DIRETORIO_ATUAL.parents) if (p / 'src' / 'pipeline.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')
print(f'Raiz do projeto: {ROOT}')
EXECUTAR_AUDITORIAS = False
EXECUTAR_PIPELINE_INDICADOR = False
pd.set_option('display.max_columns', 30)

In [ ]:
AUDITORIAS = [
 'src/auditoria_saeb/inspecionar_saeb.py',
 'src/auditoria_saeb/verificar_cabecalho_saeb.py',
 'src/auditoria_saeb/verificar_categorias_saeb.py',
 'src/auditoria_saeb/verificar_categorias_1315.py',
 'src/auditoria_saeb/verificar_codigos_2011.py',
 'src/auditoria_saeb/verificar_saeb_escola.py',
]
ETAPAS = [
 'src/bronze/saeb/ingest_saeb.py',
 'src/bronze/saeb/validar_bronze_saeb.py',
 'src/bronze/saeb/ingest_saeb_resultados_2023.py',
 'src/bronze/saeb/validar_bronze_saeb_resultados_2023.py',
 'src/silver/saeb/transformar_saeb.py',
 'src/silver/saeb/validar_silver_saeb.py',
 'src/gold/saeb/transformar_saeb.py',
 'src/gold/saeb/validar_fato_saeb.py',
]
def executar(scripts):
    for script in scripts:
        print(f'\n>>> {script}')
        subprocess.run([sys.executable, script], cwd=ROOT, check=True)
print('Auditorias:', *AUDITORIAS, sep='\n- ')
print('\nEtapas:', *ETAPAS, sep='\n- ')

In [ ]:
if EXECUTAR_AUDITORIAS:
    executar(AUDITORIAS)
else:
    print('Auditorias nao executadas.')
if EXECUTAR_PIPELINE_INDICADOR:
    executar(ETAPAS)
else:
    print('Pipeline nao executado; leitura dos artefatos existentes.')

## Evidencias por camada

Os arquivos de microdados e escola presentes na Raw servem a rastreabilidade e auditoria. A serie de consumo utiliza os resultados oficiais agregados por UF documentados no projeto.

In [ ]:
raw = sorted((ROOT / 'data/raw/saeb').glob('*'))
bronze = sorted((ROOT / 'data/bronze/saeb').glob('*.parquet'))
silver_path = ROOT / 'data/silver/saeb/saeb_2007_2023.parquet'
gold_path = ROOT / 'data/gold/fatos/fato_saeb.parquet'
pd.DataFrame({'camada':['Raw','Bronze','Silver','Gold'], 'arquivos':[len(raw),len(bronze),int(silver_path.exists()),int(gold_path.exists())], 'destino':[str(ROOT/'data/raw/saeb'),str(ROOT/'data/bronze/saeb'),str(silver_path),str(gold_path)]})

In [ ]:
historicos = [p for p in bronze if 'resultados_uf' not in p.name]
amostra_bronze = pd.read_parquet(historicos[0])
silver = pd.read_parquet(silver_path)
gold = pd.read_parquet(gold_path)
display(amostra_bronze.head(3), silver.head(3), gold.head(3))
pd.DataFrame({'camada':['Bronze (uma edicao)','Silver','Gold'], 'linhas':[len(amostra_bronze),len(silver),len(gold)], 'colunas':[len(amostra_bronze.columns),len(silver.columns),len(gold.columns)]})

### Evidencia das regras por edicao

A tabela deixa visiveis as categorias historicas de rede, localizacao e capital que foram harmonizadas para `PUBLICA`. A amostra de linhagem identifica a celula de origem de cada proficiencia.

In [ ]:
regras_edicao = (silver.groupby(['ANO','REDE_ORIGEM','LOCALIZACAO_ORIGEM','CAPITAL_ORIGEM','REDE'], as_index=False).agg(REGISTROS=('PROFICIENCIA','size'), AUSENTES=('PROFICIENCIA',lambda s: int(s.isna().sum()))))
display(regras_edicao)
linhagem = ['ANO','UF','ETAPA','DISCIPLINA','PROFICIENCIA','REDE','REDE_ORIGEM','LOCALIZACAO_ORIGEM','CAPITAL_ORIGEM','ARQUIVO_ORIGEM','ABA_ORIGEM','LINHA_ORIGEM_BRONZE','COLUNA_ORIGEM','GRANULARIDADE_ORIGEM']
display(silver.loc[:, linhagem].head(4))
pd.DataFrame({'Silver_preserva_para_auditoria':[', '.join(c for c in silver.columns if c not in gold.columns)], 'Gold_entrega_ao_BI':[', '.join(gold.columns)]})

In [ ]:
resumo = gold.groupby(['ANO','DISCIPLINA'], as_index=False)['PROFICIENCIA'].mean()
grafico = resumo.pivot(index='ANO', columns='DISCIPLINA', values='PROFICIENCIA')
ax = grafico.plot(figsize=(10,5), marker='o')
ax.set(title='Proficiencia SAEB media das UFs da rede publica', xlabel='Ano da edicao', ylabel='Proficiencia')
ax.grid(alpha=0.25)
plt.show()

## Disponibilizacao para o BI

`fato_saeb.parquet` se relaciona a `DIM_UF`, `DIM_TEMPO` e `DIM_ETAPA`. As medidas de Portugues e Matematica permanecem no arquivo DAX do projeto. A visualizacao e descritiva e nao recompõe resultados a partir de escolas.